# Week 1 Concepts — Computer Vision & Time Series

Runnable companion notebook for Days 1-4. Each section links back to its matching
daily note ([Day1](../daily/Day1_cnn-vs-vit.md), [Day2](../daily/Day2_diffusion-models.md),
[Day3](../daily/Day3_time-series-fundamentals.md), [Day4](../daily/Day4_forecasting-methods.md))
and reuses the same worked example, so the shapes and numbers printed here are the
ones referenced in the prose there.

**On execution**: the numpy/pandas/statsmodels cells below (Day 1's patch-embedding
and attention shape trace, Day 2's forward-diffusion noise schedule, all of Day 3,
and all of Day 4) were executed in a plain numpy/pandas/statsmodels environment with
no GPU, and the outputs shown under each cell are the real, unedited output from that
run. The two `transformers`/`diffusers` pipeline cells (end of Day 1, end of Day 2)
use the current, real API for those libraries but were **not** executed here, since
this environment has no GPU and no torch/transformers/diffusers installation — they
are included as accurate reference code, not as verified-to-run-as-is scripts.


## Day 1: CNNs vs. Vision Transformers

First, a from-scratch numpy shape trace through both a single conv layer and a full
ViT patch-embedding -> CLS token -> position embedding -> self-attention pipeline.
No deep learning framework required to see exactly how the tensors change shape at
every step. Then, a real (but unexecuted here) `transformers` pipeline pattern for
batch image classification with a confidence-based review flag.


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# ---- an image as a tensor ----
B, C, H, W = 1, 3, 224, 224
image = rng.uniform(0, 1, size=(B, C, H, W)).astype(np.float32)
# shape: (batch=1, channels=3, H=224, W=224)
print("image:", image.shape, image.dtype)

# ---- CNN: one conv layer, implemented directly (im2col-style) so the
# shape math is visible instead of hidden inside a library call ----
kernel = rng.normal(size=(8, C, 3, 3)).astype(np.float32)
# shape: (out_channels=8, in_channels=3, kh=3, kw=3)

def conv2d_naive(x, w, stride=1):
    B, C, H, W = x.shape
    OC, _, kh, kw = w.shape
    oh = (H - kh) // stride + 1          # output height shrinks: no padding
    ow = (W - kw) // stride + 1          # output width shrinks the same way
    out = np.zeros((B, OC, oh, ow), dtype=np.float32)
    w_flat = w.reshape(OC, -1)            # (OC, C*kh*kw) -- flatten each filter
    for i in range(oh):
        for j in range(ow):
            # grab the receptive-field patch this output position depends on
            patch = x[:, :, i*stride:i*stride+kh, j*stride:j*stride+kw]   # (B, C, kh, kw)
            patch_flat = patch.reshape(B, -1)                             # (B, C*kh*kw)
            out[:, :, i, j] = patch_flat @ w_flat.T                       # (B, OC) dot product per filter
    return out

# run on a 16x16 crop only -- a pure-python loop over the full 224x224
# image would be needlessly slow for a shape demo
small_crop = image[:, :, :16, :16]                  # shape: (1, 3, 16, 16)
feature_map = conv2d_naive(small_crop, kernel)      # shape: (1, 8, 14, 14)
print("conv feature_map:", feature_map.shape)   # (16-3)/1+1 = 14 per spatial dim

# ---- ViT: patchify + linear projection + CLS token + position embedding ----
patch_size = 16
n_patches_h = H // patch_size                       # 224 / 16 = 14
n_patches_w = W // patch_size                        # 14
n_patches = n_patches_h * n_patches_w                 # 196
patch_dim = C * patch_size * patch_size               # 3*16*16 = 768
embed_dim = 768                                        # ViT-Base hidden size

# reshape the image into a grid of non-overlapping patches, then flatten
# each patch into a single vector -- no learned weights involved yet
x = image.reshape(B, C, n_patches_h, patch_size, n_patches_w, patch_size)
x = x.transpose(0, 2, 4, 1, 3, 5)      # group each patch's pixels together
patches = x.reshape(B, n_patches, patch_dim)
print("patches:", patches.shape)  # (1, 196, 768) -- 196 patches, raw 768-length pixel vectors

W_proj = rng.normal(scale=0.02, size=(patch_dim, embed_dim)).astype(np.float32)
with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    patch_tokens = patches @ W_proj
print("patch_tokens:", patch_tokens.shape)  # (1, 196, 768) -- now a learned embedding

cls_token = rng.normal(scale=0.02, size=(1, 1, embed_dim)).astype(np.float32)
cls_tokens = np.repeat(cls_token, B, axis=0)         # shape: (1, 1, 768)
tokens = np.concatenate([cls_tokens, patch_tokens], axis=1)
print("tokens w/ CLS:", tokens.shape)  # (1, 197, 768) -- CLS token is now position 0

pos_embed = rng.normal(scale=0.02, size=(1, n_patches + 1, embed_dim)).astype(np.float32)
tokens = tokens + pos_embed
print("tokens + pos_embed:", tokens.shape)  # (1, 197, 768) -- same shape, position now encoded in values

# ---- one self-attention layer over the full 197-token sequence ----
Wq = rng.normal(scale=0.02, size=(embed_dim, embed_dim)).astype(np.float32)
Wk = rng.normal(scale=0.02, size=(embed_dim, embed_dim)).astype(np.float32)
Wv = rng.normal(scale=0.02, size=(embed_dim, embed_dim)).astype(np.float32)

with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    Q = tokens @ Wq   # shape: (1, 197, 768) -- one query vector per token
    K = tokens @ Wk   # shape: (1, 197, 768) -- one key vector per token
    V = tokens @ Wv   # shape: (1, 197, 768) -- one value vector per token

    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(embed_dim)
    # shape: (1, 197, 197) -- scores[0, i, j] = how much token i attends to token j
    print("attention scores:", scores.shape)

    scores = scores - scores.max(axis=-1, keepdims=True)   # numerical stability
    attn = np.exp(scores)
    attn = attn / attn.sum(axis=-1, keepdims=True)          # rows sum to 1
    out = attn @ V
print("attention output:", out.shape)  # (1, 197, 768) -- weighted blend of every token's value vector

print("row sums ~1:", round(float(attn[0, 0].sum()), 6))
print("CLS -> last patch weight:", float(attn[0, 0, -1]))
print("CLS -> first patch weight:", float(attn[0, 0, 1]))


image: (1, 3, 224, 224) float32
conv feature_map: (1, 8, 14, 14)
patches: (1, 196, 768)
patch_tokens: (1, 196, 768)
tokens w/ CLS: (1, 197, 768)
tokens + pos_embed: (1, 197, 768)
attention scores: (1, 197, 197)
attention output: (1, 197, 768)
row sums ~1: 1.0
CLS -> last patch weight: 0.005082810592993156
CLS -> first patch weight: 0.005071407237220381


### Reference pattern: batch classification with `transformers` (not executed here)

Real, current `transformers` API. Requires `torch` and a downloaded checkpoint, so it
is not run in this notebook -- but the tensor shapes follow directly from the ViT
walkthrough above: `pixel_values` is `(1, 3, 224, 224)`, and `logits` is
`(1, num_labels)` for whichever classification head the checkpoint uses.


In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import torch
import pandas as pd

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")
model.eval()  # disable dropout etc. -- this is inference, not training

camera_trap_photos = ["trailcam_0091.jpg", "trailcam_0092.jpg", "trailcam_0093.jpg"]

rows = []
for path in camera_trap_photos:
    image = Image.open(path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    # inputs["pixel_values"] shape: (1, 3, 224, 224)

    with torch.no_grad():  # no backward pass needed for inference
        logits = model(**inputs).logits
    # logits shape: (1, 1000) for this ImageNet-1k checkpoint

    probs = logits.softmax(dim=-1)[0]        # shape: (1000,), sums to 1.0
    top_prob, top_idx = probs.max(dim=-1)

    rows.append({
        "file": path,
        "predicted_label": model.config.id2label[top_idx.item()],
        "confidence": round(top_prob.item(), 3),
        "needs_review": top_prob.item() < 0.6,
    })

results = pd.DataFrame(rows)  # -> DataFrame, columns [file, predicted_label, confidence, needs_review], len=3
results


## Day 2: Diffusion Models and Text-to-Image Generation

First, a from-scratch numpy trace of the forward-diffusion noise schedule (how
signal-to-noise decays across timesteps) and the cross-attention shape math that
lets a U-Net's spatial locations attend to text token embeddings. Then, a real (but
unexecuted here) `diffusers` pipeline pattern comparing `num_inference_steps` and
`guidance_scale` settings.


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# ---- a tiny "latent" standing in for a VAE-encoded image ----
# Real Stable-Diffusion latents are (batch, 4, 64, 64) for a 512x512 image;
# we use the same shape here so the arithmetic is representative.
B, C, Hl, Wl = 1, 4, 64, 64
x0 = rng.uniform(-1, 1, size=(B, C, Hl, Wl)).astype(np.float32)
print("x0 (clean latent):", x0.shape, "mean", round(float(x0.mean()), 4), "std", round(float(x0.std()), 4))

# ---- fixed linear noise schedule (no learning involved) ----
T = 1000
betas = np.linspace(1e-4, 0.02, T).astype(np.float32)   # shape: (1000,)
alphas = 1.0 - betas                                      # shape: (1000,)
alpha_bars = np.cumprod(alphas)                            # shape: (1000,), monotonically decreasing

def forward_diffuse(x0, t_index, alpha_bars, rng):
    """Closed-form sample of x_t given x0, skipping every intermediate step."""
    a_bar = alpha_bars[t_index]
    noise = rng.normal(size=x0.shape).astype(np.float32)   # shape matches x0
    xt = np.sqrt(a_bar) * x0 + np.sqrt(1 - a_bar) * noise    # shape matches x0
    return xt, noise

print("\nt      xt.shape          signal_scale  noise_scale")
for t_index in [0, 249, 499, 999]:
    xt, noise = forward_diffuse(x0, t_index, alpha_bars, rng)
    signal_scale = float(np.sqrt(alpha_bars[t_index]))
    noise_scale = float(np.sqrt(1 - alpha_bars[t_index]))
    print(f"{t_index:4d}   {str(xt.shape):16s}  {signal_scale:.4f}        {noise_scale:.4f}")

# ---- cross-attention shape trace: latent spatial tokens attend to text tokens ----
n_spatial_tokens = Hl * Wl                      # 64*64 = 4096
latent_seq = x0.reshape(B, C, n_spatial_tokens).transpose(0, 2, 1)  # (B, 4096, 4)
print("\nlatent flattened to sequence:", latent_seq.shape)

n_text_tokens, text_dim = 77, 768  # CLIP's fixed context length x embedding dim
text_embeddings = rng.normal(scale=0.02, size=(B, n_text_tokens, text_dim)).astype(np.float32)
print("text_embeddings:", text_embeddings.shape)

attn_dim = 128
Wq = rng.normal(scale=0.02, size=(C, attn_dim)).astype(np.float32)
Wk = rng.normal(scale=0.02, size=(text_dim, attn_dim)).astype(np.float32)
Wv = rng.normal(scale=0.02, size=(text_dim, attn_dim)).astype(np.float32)

with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    Q = latent_seq @ Wq          # (B, 4096, attn_dim) -- one query per spatial location
    K = text_embeddings @ Wk     # (B, 77, attn_dim)   -- one key per text token
    V = text_embeddings @ Wv     # (B, 77, attn_dim)
    print("Q (spatial queries):", Q.shape)
    print("K, V (text keys/values):", K.shape, V.shape)

    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(attn_dim)  # (B, 4096, 77)
    print("cross-attention scores:", scores.shape)  # each of 4096 latent pixels scores all 77 text tokens

    scores = scores - scores.max(axis=-1, keepdims=True)
    attn = np.exp(scores)
    attn = attn / attn.sum(axis=-1, keepdims=True)
    cross_out = attn @ V  # (B, 4096, attn_dim)
print("cross-attention output:", cross_out.shape)  # per-pixel text-conditioned features


x0 (clean latent): (1, 4, 64, 64) mean 0.0034 std 0.5764

t      xt.shape          signal_scale  noise_scale
   0   (1, 4, 64, 64)    0.9999        0.0100
 249   (1, 4, 64, 64)    0.7239        0.6899
 499   (1, 4, 64, 64)    0.2803        0.9599
 999   (1, 4, 64, 64)    0.0064        1.0000

latent flattened to sequence: (1, 4096, 4)
text_embeddings: (1, 77, 768)
Q (spatial queries): (1, 4096, 128)
K, V (text keys/values): (1, 77, 128) (1, 77, 128)
cross-attention scores: (1, 4096, 77)
cross-attention output: (1, 4096, 128)


### Reference pattern: batch generation with `diffusers` (not executed here)

Real, current `diffusers` API. Requires `torch` and `diffusers`, and a GPU is
strongly recommended, so it is not run in this notebook -- but every shape in the
loop follows from the pipeline mechanics covered in the Day 2 note: a `(1, 4, 64, 64)`
latent gets denoised over `num_inference_steps` U-Net calls, then decoded by the VAE
into a `(1, 3, 512, 512)`-equivalent output image.


In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import pandas as pd

pipe = StableDiffusionPipeline.from_pretrained(
    "segmind/small-sd",          # distilled, lightweight checkpoint
    torch_dtype=torch.float32,   # float32 for CPU; float16 typically used on GPU
)

sticker_prompts = [
    "a cartoon sticker of a sleepy cat wearing headphones, flat vector style",
    "a cartoon sticker of a rocket ship with a smiling face, flat vector style",
]

rows = []
for prompt in sticker_prompts:
    for steps, guidance in [(15, 4.0), (30, 7.5), (30, 12.0)]:
        # each call runs the full reverse-diffusion loop: `steps` sequential
        # U-Net evaluations denoising a 64x64x4 latent, then one VAE decode
        image = pipe(prompt, num_inference_steps=steps, guidance_scale=guidance).images[0]
        fname = f"sticker_{sticker_prompts.index(prompt)}_{steps}_{guidance}.png"
        image.save(fname)
        rows.append({"prompt": prompt, "steps": steps, "guidance_scale": guidance, "file": fname})

comparison = pd.DataFrame(rows)  # -> DataFrame, len = 2 prompts * 3 settings = 6
comparison


## Day 3: Time Series Fundamentals

A synthetic two-year daily "foot traffic" series (trend + weekly seasonality +
noise) -- the same series used throughout the Day 3 and Day 4 notes. Below:
autocorrelation on the raw series vs. the detrended series (showing how a strong
trend inflates every lag and hides the real seasonal peak), resampling, a full
`seasonal_decompose` call, and then anomaly detection on raw vs. deseasonalized
data to see the seasonal-peak false-positive problem directly -- including
injecting one genuine anomaly and confirming only that one gets flagged.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf

rng = np.random.default_rng(7)

# ---- synthetic daily foot-traffic series: trend + weekly seasonality + noise ----
dates = pd.date_range("2023-01-01", periods=730, freq="D")    # 2 years of daily data
trend = np.linspace(80, 160, len(dates))                       # slow linear growth
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)           # period-7 seasonal wave
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")
print("foot_traffic:", foot_traffic.shape, foot_traffic.dtype)  # -> Series, len=730, float64

# ---- autocorrelation on the RAW series: trend inflates every lag ----
acf_raw = acf(foot_traffic, nlags=14)
print("\nACF on raw series (trend inflates every lag):")
for lag in [0, 1, 6, 7, 8, 14]:
    marker = "  <-- local peak" if lag in (7, 14) else ""
    print(f"  lag {lag:2d}: {acf_raw[lag]:+.3f}{marker}")

# ---- autocorrelation on the DETRENDED series: seasonal peak becomes obvious ----
trend_est = seasonal_decompose(foot_traffic, model="additive", period=7).trend
detrended = (foot_traffic - trend_est).dropna()
acf_detrended = acf(detrended, nlags=14)
print("\nACF on detrended series (lag-7 is now the clear peak):")
for lag in [0, 1, 3, 7, 14]:
    marker = "  <-- clear peak" if lag == 7 else ""
    print(f"  lag {lag:2d}: {acf_detrended[lag]:+.3f}{marker}")

# ---- resampling: downsample daily -> weekly ----
weekly_avg = foot_traffic.resample("W").mean()
print("\nweekly_avg:", weekly_avg.shape)  # daily len=730 -> weekly len ~104-106

# ---- full decomposition ----
result = seasonal_decompose(foot_traffic, model="additive", period=7)
print("\nresult.trend:", result.trend.shape, result.trend.dtype)
print("result.seasonal:", result.seasonal.shape)
print("result.resid:", result.resid.shape)
print("NaNs in trend (centered window edge effect):", int(result.trend.isna().sum()))
print("seasonal component, first 14 values (repeats every 7):")
print(result.seasonal.head(14).round(2).tolist())


foot_traffic: (730,) float64

ACF on raw series (trend inflates every lag):
  lag  0: +1.000
  lag  1: +0.913
  lag  6: +0.897
  lag  7: +0.953  <-- local peak
  lag  8: +0.888
  lag 14: +0.930  <-- local peak

ACF on detrended series (lag-7 is now the clear peak):
  lag  0: +1.000
  lag  1: +0.544
  lag  3: -0.833
  lag  7: +0.893  <-- clear peak
  lag 14: +0.885

weekly_avg: (106,)

result.trend: (730,) float64
result.seasonal: (730,)
result.resid: (730,)
NaNs in trend (centered window edge effect): 6
seasonal component, first 14 values (repeats every 7):
[-11.4, 0.1, 11.28, 14.41, 6.87, -6.81, -14.44, -11.4, 0.1, 11.28, 14.41, 6.87, -6.81, -14.44]


Now, anomaly detection: a naive rolling 3-sigma detector on the raw series vs. on
the deseasonalized residual, plus one genuinely injected anomaly to confirm the
residual-based version actually catches the real thing and nothing else.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

rng = np.random.default_rng(7)
dates = pd.date_range("2023-01-01", periods=730, freq="D")
trend = np.linspace(80, 160, len(dates))
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")
result = seasonal_decompose(foot_traffic, model="additive", period=7)

# ---- anomaly check on the RAW series: rolling window mixes weekday/weekend baselines ----
roll_mean_raw = foot_traffic.rolling(14).mean()
roll_std_raw = foot_traffic.rolling(14).std()
raw_anomaly = (foot_traffic - roll_mean_raw).abs() > 3 * roll_std_raw
print("Raw-series anomalies flagged (rolling 14d, 3-sigma):", int(raw_anomaly.sum()))

# ---- anomaly check on the DESEASONALIZED residual ----
resid = result.resid.dropna()
roll_mean_resid = resid.rolling(14).mean()
roll_std_resid = resid.rolling(14).std()
resid_anomaly = (resid - roll_mean_resid).abs() > 3 * roll_std_resid
print("Deseasonalized-residual anomalies flagged:", int(resid_anomaly.sum()))

# ---- inject one genuine one-day anomaly and confirm it's the only thing flagged ----
foot_traffic_with_spike = foot_traffic.copy()
foot_traffic_with_spike.iloc[400] += 60   # e.g. a local event, a viral social post
result2 = seasonal_decompose(foot_traffic_with_spike, model="additive", period=7)
resid2 = result2.resid.dropna()
roll_mean2 = resid2.rolling(14).mean()
roll_std2 = resid2.rolling(14).std()
anomaly2 = (resid2 - roll_mean2).abs() > 3 * roll_std2
flagged_dates = resid2.index[anomaly2]
print("\nWith an injected spike at index 400:")
print("  flagged dates:", [d.strftime('%Y-%m-%d') for d in flagged_dates])
print("  actual spike date:", foot_traffic.index[400].strftime("%Y-%m-%d"))


Raw-series anomalies flagged (rolling 14d, 3-sigma): 0
Deseasonalized-residual anomalies flagged: 0

With an injected spike at index 400:
  flagged dates: ['2024-02-05']
  actual spike date: 2024-02-05


## Day 4: Forecasting Methods and Evaluation

Same foot-traffic series, split at a chronological cutoff (last 60 days held out).
First, the Augmented Dickey-Fuller stationarity test on the raw vs. differenced
series. Then the full ladder of forecasting methods -- moving average, simple
exponential smoothing, Holt, Holt-Winters -- scored against the held-out window
with MAE / MAPE / RMSE. Finally, a simulated prediction interval instead of a
single point forecast, checked against how often the true values actually fell
inside the band.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller

rng = np.random.default_rng(7)

# ---- same synthetic foot-traffic series as Day 3 ----
dates = pd.date_range("2023-01-01", periods=730, freq="D")
trend = np.linspace(80, 160, len(dates))
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")

train, test = foot_traffic[:-60], foot_traffic[-60:]
print("train:", train.shape, "test:", test.shape)  # -> (670,) (60,)

# ---- stationarity check: ADF test on raw vs. differenced series ----
stat, p_value, *_ = adfuller(foot_traffic)
print(f"\nADF on raw series:        stat={stat:.3f}  p={p_value:.4f}  "
      f"({'stationary' if p_value < 0.05 else 'NOT stationary'})")

diffed = foot_traffic.diff().dropna()   # -> Series, len=729 (one point lost to differencing)
stat_d, p_value_d, *_ = adfuller(diffed)
print(f"ADF on differenced series: stat={stat_d:.3f}  p={p_value_d:.6f}  "
      f"({'stationary' if p_value_d < 0.05 else 'NOT stationary'})")


train: (670,) test: (60,)

ADF on raw series:        stat=-0.568  p=0.8780  (NOT stationary)
ADF on differenced series: stat=-10.298  p=0.000000  (stationary)


The forecasting ladder, fit and scored end to end:

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

rng = np.random.default_rng(7)
dates = pd.date_range("2023-01-01", periods=730, freq="D")
trend = np.linspace(80, 160, len(dates))
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")
train, test = foot_traffic[:-60], foot_traffic[-60:]

# ---- the ladder of methods, from naive to seasonal ----
def moving_average_forecast(train, horizon, window=7):
    last_avg = train.iloc[-window:].mean()
    return pd.Series([last_avg] * horizon, index=test.index)  # flat line, no trend/season

ma_forecast = moving_average_forecast(train, len(test))

ses_model = ExponentialSmoothing(train, trend=None, seasonal=None).fit()
ses_forecast = ses_model.forecast(len(test))

holt_model = ExponentialSmoothing(train, trend="add", seasonal=None).fit()
holt_forecast = holt_model.forecast(len(test))

hw_model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=7).fit()
hw_forecast = hw_model.forecast(len(test))
print("hw_forecast:", hw_forecast.shape)  # -> Series, len=60

# ---- score every method against the held-out test set ----
def mae(y, yhat): return float(np.mean(np.abs(y - yhat)))
def mape(y, yhat): return float(np.mean(np.abs((y - yhat) / y)) * 100)
def rmse(y, yhat): return float(np.sqrt(np.mean((y - yhat) ** 2)))

print(f"\n{'Method':16s} {'MAE':>6s}  {'MAPE(%)':>7s}  {'RMSE':>6s}")
for name, fc in [("MovingAvg", ma_forecast), ("SES", ses_forecast),
                  ("Holt", holt_forecast), ("Holt-Winters", hw_forecast)]:
    print(f"{name:16s} {mae(test, fc):6.2f}  {mape(test, fc):6.2f}   {rmse(test, fc):6.2f}")


hw_forecast: (60,)

Method              MAE  MAPE(%)    RMSE
MovingAvg          9.80    6.34    11.46
SES               10.17    6.66    11.87
Holt              11.29    7.53    13.41
Holt-Winters       3.32    2.12     4.17


A range instead of a single number: simulate many plausible paths and report a
90% band, then check empirically how often the truth actually landed inside it.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

rng = np.random.default_rng(7)
dates = pd.date_range("2023-01-01", periods=730, freq="D")
trend = np.linspace(80, 160, len(dates))
weekly = 15 * np.sin(2 * np.pi * dates.dayofweek / 7)
noise = rng.normal(0, 4, len(dates))
foot_traffic = pd.Series(trend + weekly + noise, index=dates, name="visitors")
train, test = foot_traffic[:-60], foot_traffic[-60:]

hw_model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=7).fit()

# ---- prediction interval via simulation, instead of a single point forecast ----
simulations = hw_model.simulate(len(test), repetitions=200, error="add", random_state=7)
print("simulations:", simulations.shape)  # -> DataFrame, (60, 200): 60 steps x 200 sampled paths

lower = simulations.quantile(0.05, axis=1)   # Series, len=60 -- 5th percentile per day
upper = simulations.quantile(0.95, axis=1)   # Series, len=60 -- 95th percentile per day
print("lower/upper band shape:", lower.shape, upper.shape)

in_band = ((test.values >= lower.values) & (test.values <= upper.values)).mean()
print(f"fraction of true test points inside the 90% band: {in_band:.2%}")


simulations: (60, 200)
lower/upper band shape: (60,) (60,)
fraction of true test points inside the 90% band: 83.33%
